In [1]:
import re
import string
import emoji
import numpy as np
from collections import Counter, defaultdict
from sklearn.cluster import AgglomerativeClustering
import phonetics

In [2]:
def load_corpus(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    return text

file_path = "../data_text_files/gpt_cluster_labels.txt"  
text = load_corpus(file_path)
text

"\nCluster Labels:\nCluster 0 -> teslai: ['teslai', 'tesle']\nCluster 1 -> kathaa: ['kathaa', 'chhaithau']\nCluster 2 -> bhitra: ['bhitra', 'bitra']\nCluster 3 -> pani: ['bhan', 'pugna', 'piunu', 'bhaane', 'piuney', 'pani', 'piuna', 'bhaannu', 'bhanya', 'pugnu', 'piune', 'bhannu', 'bhayena', 'pugne', 'bhane', 'bhaena']\nCluster 4 -> bolchha: ['bolchha']\nCluster 5 -> hidney: ['hidney', 'hidnu', 'hidna', 'hidne']\nCluster 6 -> aaucha: ['aaucha']\nCluster 7 -> samjhinchhan: ['samjhinchhan']\nCluster 8 -> khulne: ['khulne', 'khulna', 'khulnu', 'khelnu']\nCluster 9 -> suttai: ['suttai', 'sodhiyo', 'sutta', 'seto', 'saatou', 'sodhe']\nCluster 10 -> raat: ['raat', 'raato', 'roti']\nCluster 11 -> uttar: ['utaryo', 'yatra', 'yatraa', 'uttar']\nCluster 12 -> uniharu: ['uniharu']\nCluster 13 -> lagna: ['lagna', 'launa', 'launu', 'laune', 'launey', 'lagne', 'lagnu']\nCluster 14 -> dekhera: ['dekhera']\nCluster 15 -> tala: ['tala', 'tali']\nCluster 16 -> thiyekocha: ['thiyekocha']\nCluster 17 -> h

In [ ]:
def clean_text(text):
    # Remove emojis
    text = emoji.replace_emoji(text, replace='')
    # Lowercase
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    # Remove numbers
    text = re.sub(r"\d+", " ", text)
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean = clean_text(text)
clean

13782

In [21]:
def get_unique_words(text):
    return list(set(text.split()))

words = get_unique_words(clean)
len(words)

775

In [3]:
from nepPhoneticEncoder import nepali_dmetaphone
import jellyfish
def get_phonetic_codes(words):
    phonetic_map = {}
    for w in words:
        codes,a  =  nepali_dmetaphone(w)
        #codes = jellyfish.soundex(w)
        # Take only first non-None code for clustering
        full_code = ''.join(c for c in codes if c is not None)
        if full_code:
            phonetic_map[w] = full_code
    return phonetic_map

words  = ["hudeina", "hudena", "heray", "heramla", "herau", "herara", "herum", "hereraw", "herei", "herxam", "heresi", "hernush"]
phonetic_map = get_phonetic_codes(words)

phonetic_map

Preprocessed: hudina
Tokens: ['h', 'u', 'd', 'i', 'n', 'a']
Primary: HDN
Alternate: HDN
Preprocessed: hudena
Tokens: ['h', 'u', 'd', 'e', 'n', 'a']
Primary: HDN
Alternate: HDN
Preprocessed: heray
Tokens: ['h', 'e', 'r', 'a', 'y']
Primary: HRY
Alternate: HRY
Preprocessed: herala
Tokens: ['h', 'e', 'r', 'a', 'l', 'a']
Primary: HRL
Alternate: HRL
Preprocessed: herau
Tokens: ['h', 'e', 'r', 'a', 'u']
Primary: HR
Alternate: HR
Preprocessed: herara
Tokens: ['h', 'e', 'r', 'a', 'r', 'a']
Primary: HR
Alternate: HR
Preprocessed: heru
Tokens: ['h', 'e', 'r', 'u']
Primary: HR
Alternate: HR
Preprocessed: hererab
Tokens: ['h', 'e', 'r', 'e', 'r', 'a', 'b']
Primary: HRB
Alternate: HRB
Preprocessed: heri
Tokens: ['h', 'e', 'r', 'i']
Primary: HR
Alternate: HR
Preprocessed: herXa
Tokens: ['h', 'e', 'r', 'X', 'a']
Primary: HRK
Alternate: HRX
Preprocessed: heresi
Tokens: ['h', 'e', 'r', 'e', 's', 'i']
Primary: HRS
Alternate: HRS
Preprocessed: hernuSH
Tokens: ['h', 'e', 'r', 'n', 'u', 'SH']
Primary: HRNS


{'hudeina': 'HDN',
 'hudena': 'HDN',
 'heray': 'HRY',
 'heramla': 'HRL',
 'herau': 'HR',
 'herara': 'HR',
 'herum': 'HR',
 'hereraw': 'HRB',
 'herei': 'HR',
 'herxam': 'HRK',
 'heresi': 'HRS',
 'hernush': 'HRNS'}

In [14]:
def phonetic_to_features(phonetic_map):
    # Simple way: map each char to int (A=1, B=2...) and pad
    max_len = max(len(c) for c in phonetic_map.values())
    features = []
    words_list = []
    for word, code in phonetic_map.items():
        vec = [ord(ch) - 64 for ch in code]  # A=1, B=2...
        vec += [0] * (max_len - len(vec))  # padding
        features.append(vec)
        words_list.append(word)
    return np.array(features), words_list


features, words_list = phonetic_to_features(phonetic_map)


for i, word in enumerate(words_list[:10]):
    print(f"{word}: {features[i]}")

diyeko: [  4 -14 -16 -16]
roti: [ 18 -13 -16 -16]
mitho: [ 13 -13 -16 -16]
aunu: [  1 -11 -16 -16]
kaamgarna: [ 11 -11 -14 -10]
kaamgarda: [ 11 -11 -14 -10]
padhda: [ 16 -13 -16 -16]
kinaabhayo: [ 11 -11 -15 -16]
laune: [ 12 -11 -16 -16]
chito: [  3 -13 -16 -16]


In [15]:
from collections import defaultdict

def dynamic_phonetic_clustering(phonetic_map):
    clusters = []  # list of clusters, each cluster is a list of phonetic codes
    cluster_words_map = defaultdict(list)  # cluster idx -> list of words
    
    for word, code in phonetic_map.items():
        added = False
        # Check existing clusters for a match
        for idx, cluster_codes in enumerate(clusters):
            # If any phonetic code in the cluster matches the current code
            if any(c == code for c in cluster_codes):
                cluster_codes.append(code)
                cluster_words_map[idx].append(word)
                added = True
                break
        
        # If no match found, create a new cluster
        if not added:
            clusters.append([code])
            cluster_words_map[len(clusters)-1].append(word)
    
    return cluster_words_map

cluster_map = dynamic_phonetic_clustering(phonetic_map)
print(len(cluster_map))
print(cluster_map)

297
defaultdict(<class 'list'>, {0: ['diyeko', 'dekhiyo', 'dasau', 'dekhe', 'dekhchha'], 1: ['roti', 'raato', 'raat'], 2: ['mitho', 'maathi', 'mithai', 'mathi'], 3: ['aunu', 'aama', 'aayena', 'aaunu'], 4: ['kaamgarna', 'kaamgarda', 'kaamnagarna', 'kaamgarera', 'kaamgare', 'kaamgarchhan', 'kaamgareko', 'kaamgarchha'], 5: ['padhda', 'piuthiyo', 'pahad', 'path', 'pet', 'padhthiyo'], 6: ['kinaabhayo', 'kinaba', 'kinaaba'], 7: ['laune', 'launu', 'launey', 'launa'], 8: ['chito', 'chhaithau', 'choutho'], 9: ['uththa', 'uththiyo', 'uththaa', 'uthiyo'], 10: ['prayoggardai', 'prayoggarera', 'prayoggareko', 'prayoggarchhan', 'prayoggarda', 'prayoggarna', 'prayoggare', 'prayoggarchha'], 11: ['pani', 'piuney', 'piune', 'piunu', 'piuna'], 12: ['paanchau', 'piunecha'], 13: ['dinera', 'dinharu', 'dhunera'], 14: ['truck'], 15: ['chhori', 'chhoro'], 16: ['aaucha', 'aauchha', 'aayeko', 'aaja'], 17: ['padhchhan'], 18: ['pugthena', 'pakthena'], 19: ['dhunchhan', 'dinchhan', 'dhunechhan'], 20: ['rakhe', 'ro

In [16]:
def get_cluster_labels(cluster_map, corpus_text):
    corpus_words = corpus_text.split()
    word_counts = Counter(corpus_words)
    cluster_labels = {}
    for cluster_id, words in cluster_map.items():
        most_common_word = max(words, key=lambda w: word_counts[w])
        cluster_labels[cluster_id] = most_common_word
    return cluster_labels

cluster_labels = get_cluster_labels(cluster_map, clean)

In [17]:
def find_cluster_for_word(word, phonetic_map, cluster_labels, cluster_map):
    if word not in phonetic_map:
        return None
    for cluster_id, words in cluster_map.items():
        if word in words:
            return cluster_id, cluster_labels[cluster_id]
    return None

In [18]:
print("\nCluster Labels:")
for cid, label in cluster_labels.items():
    print(f"Cluster {cid} -> {label}: {cluster_map[cid]}")



Cluster Labels:
Cluster 0 -> dekhe: ['diyeko', 'dekhiyo', 'dasau', 'dekhe', 'dekhchha']
Cluster 1 -> raat: ['roti', 'raato', 'raat']
Cluster 2 -> mathi: ['mitho', 'maathi', 'mithai', 'mathi']
Cluster 3 -> aama: ['aunu', 'aama', 'aayena', 'aaunu']
Cluster 4 -> kaamgarna: ['kaamgarna', 'kaamgarda', 'kaamnagarna', 'kaamgarera', 'kaamgare', 'kaamgarchhan', 'kaamgareko', 'kaamgarchha']
Cluster 5 -> padhda: ['padhda', 'piuthiyo', 'pahad', 'path', 'pet', 'padhthiyo']
Cluster 6 -> kinaaba: ['kinaabhayo', 'kinaba', 'kinaaba']
Cluster 7 -> laune: ['laune', 'launu', 'launey', 'launa']
Cluster 8 -> chito: ['chito', 'chhaithau', 'choutho']
Cluster 9 -> uththaa: ['uththa', 'uththiyo', 'uththaa', 'uthiyo']
Cluster 10 -> prayoggardai: ['prayoggardai', 'prayoggarera', 'prayoggareko', 'prayoggarchhan', 'prayoggarda', 'prayoggarna', 'prayoggare', 'prayoggarchha']
Cluster 11 -> pani: ['pani', 'piuney', 'piune', 'piunu', 'piuna']
Cluster 12 -> paanchau: ['paanchau', 'piunecha']
Cluster 13 -> dhunera: ['di

In [19]:
# Assuming cluster_labels and cluster_map are already defined
output_file = "gpt_cluster_labels_text_soundex.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("\nCluster Labels:\n")
    for cid, label in sorted(cluster_labels.items(), key=lambda x: x[1]):
        f.write(f"Cluster {cid} -> {label}: {cluster_map[cid]}\n")

print(f"Cluster labels saved to {output_file}")

Cluster labels saved to gpt_cluster_labels_text_soundex.txt
